### Determine current BTC market

In [67]:
from time import time
from datetime import datetime
from zoneinfo import ZoneInfo
from datetime import timezone

WINDOW_SECS = 300  # 5-min window

def current_window_start() -> int:
    """
    Unix timestamp of the current 5-min window's start.
    """
    return (int(time()) // WINDOW_SECS) * WINDOW_SECS


def to_EST(ts: int) -> str:
    dt = datetime.fromtimestamp(ts, tz=timezone.utc).astimezone(ZoneInfo("America/New_York"))
    return dt.strftime("%Y-%m-%d %I:%M:%S %p EST")


def current_window_slug(ts) -> str:
    """
    Current 5-min window slug.
    """
    return f"btc-updown-5m-{ts}"


start = current_window_start()
start_EST = to_EST(start)
slug = current_window_slug(start)
print([start, start_EST, slug])

[1773848100, '2026-03-18 11:35:00 AM EST', 'btc-updown-5m-1773848100']


### Init "clob" Client

In [63]:

from py_clob_client.client import ClobClient
from config import load_config

config = load_config()

host = "https://clob.polymarket.com"
chain_id = 137  # Polygon mainnet

# Derive API credentials (L1 → L2 auth)
temp_client = ClobClient(host, key=config.polymarket.private_key, chain_id=chain_id)
api_creds = temp_client.create_or_derive_api_creds()

# Initialize trading client
client = ClobClient(
    host,
    key=config.polymarket.private_key,
    chain_id=chain_id,
    creds=api_creds,
    # I'm going with the proxy wallet through polymarket to avoid paying gas fees. This seemed like the best
    # one to use from: https://docs.polymarket.com/trading/overview#signature-types.
    signature_type=1,
    funder=config.polymarket.wallet_address,
)
client

### Get active BTC up / down Market

In [69]:
from requests import get as GET
from json import loads
from requests.exceptions import HTTPError

class MarketNotFound(Exception):
    ...

def get_market_by_slug(slug: str) -> dict:
    """Fetch a single BTC 5-min market by its exact slug."""
    response = GET(f"https://gamma-api.polymarket.com/events", params={"slug": slug}, timeout=10)
    try:
        response.raise_for_status()
        data = response.json()
        if data is None or not isinstance(data, list) or len(data) < 1:
            raise MarketNotFound(f"No market data found for slug: {slug}.")
        return data[0]
    except HTTPError as e:
        raise MarketNotFound from e
    

market = get_market_by_slug(slug)
if len(market["markets"]) != 1:
    raise AssertionError("Expected BTC Up/Down market response to contain exactly 1 market!")
outcomes = loads(market["markets"][0]["outcomes"])
tids = loads(market["markets"][0]["clobTokenIds"])
market_clobs = dict(zip([x.lower() for x in outcomes], tids))
market

{'id': '279967',
 'ticker': 'btc-updown-5m-1773848100',
 'slug': 'btc-updown-5m-1773848100',
 'title': 'Bitcoin Up or Down - March 18, 11:35AM-11:40AM ET',
 'description': 'This market will resolve to "Up" if the Bitcoin price at the end of the time range specified in the title is greater than or equal to the price at the beginning of that range. Otherwise, it will resolve to "Down".\nThe resolution source for this market is information from Chainlink, specifically the BTC/USD data stream available at https://data.chain.link/streams/btc-usd.\nPlease note that this market is about the price according to Chainlink data stream BTC/USD, not according to other sources or spot markets.',
 'resolutionSource': 'https://data.chain.link/streams/btc-usd',
 'startDate': '2026-03-17T15:43:39.56641Z',
 'creationDate': '2026-03-17T15:43:39.566406Z',
 'endDate': '2026-03-18T15:40:00Z',
 'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png',
 'icon': 'https://polymarket-uploa

In [65]:
up_book = client.get_order_book(market_clobs["down"])
up_book.asks

[OrderSummary(price='0.99', size='9933.09'),
 OrderSummary(price='0.98', size='4545.5'),
 OrderSummary(price='0.97', size='3125.65'),
 OrderSummary(price='0.96', size='2754.5'),
 OrderSummary(price='0.95', size='1891.34'),
 OrderSummary(price='0.94', size='1950.67'),
 OrderSummary(price='0.93', size='1165'),
 OrderSummary(price='0.92', size='1025'),
 OrderSummary(price='0.91', size='410'),
 OrderSummary(price='0.9', size='374'),
 OrderSummary(price='0.89', size='1104'),
 OrderSummary(price='0.88', size='445'),
 OrderSummary(price='0.87', size='255'),
 OrderSummary(price='0.86', size='893'),
 OrderSummary(price='0.85', size='288.33'),
 OrderSummary(price='0.84', size='420'),
 OrderSummary(price='0.83', size='755'),
 OrderSummary(price='0.82', size='270'),
 OrderSummary(price='0.81', size='255'),
 OrderSummary(price='0.8', size='275'),
 OrderSummary(price='0.79', size='670'),
 OrderSummary(price='0.78', size='271.2'),
 OrderSummary(price='0.77', size='260'),
 OrderSummary(price='0.76', s

In [66]:
from py_clob_client.clob_types import OrderType
from time import sleep

def in_buy_threshold(price: float, min=0.9, max=0.95) -> bool:
    return price >= min and price <= max

while True:
    up_price = client.calculate_market_price(
        token_id=market_clobs["up"],
        side="BUY",
        amount=10,
        order_type=OrderType.FOK,  # type: ignore
    )
    down_price = client.calculate_market_price(
        token_id=market_clobs["down"],
        side="BUY",
        amount=10,
        order_type=OrderType.FOK,  # type: ignore
    )
    if in_buy_threshold(up_price):
        ...
    elif in_buy_threshold(down_price):
        ...
    print({"Up": estimated_up_price, "Down": estimated_down_price})
    sleep(0.1)

NameError: name 'estimated_up_price' is not defined